# Porto Seguro Safe Driver Prediction – 02 Data Preparation

**Goal:** turn the decisions of the EDA log (notebook 01, section 11) into preprocessing pipelines, and set up
the evaluation protocol used to compare all models.

**Workflow**
1. Setup: imports and shared project code
2. Load data, recode missing values, split (identical to notebook 01)
3. Evaluation setup: 5-fold stratified cross-validation on the training part
4. Dummy baseline

## 1. Setup

In [1]:
# 1. Imports and shared project code
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier

# shared project code lives in ../src (same loading and split for every notebook)
sys.path.insert(0, str(Path.cwd().parent))
from src.data_utils import (RANDOM_STATE, load_data, split_features_target,
                            train_test_parts, feature_groups)

## 2. Load data, recode missing values and split

In [2]:
# 2. Load data (-1 recoded as NaN) and the stratified 80/20 split, shared with notebook 01
data = load_data()
features, target = split_features_target(data)
X_train, X_test, y_train, y_test = train_test_parts(features, target)

# check: must give the same numbers as in notebook 01
print(f"training part: {len(y_train):,} rows, claim rate {y_train.mean():.4%}")
print(f"test part:     {len(y_test):,} rows, claim rate {y_test.mean():.4%}")

training part: 476,169 rows, claim rate 3.6447%
test part:     119,043 rows, claim rate 3.6449%


In [3]:
# 2. Feature lists by type (from the column names, see notebook 01, section 7)
column_groups = feature_groups(features.columns)
bin_cols = column_groups['binary']
cat_cols = column_groups['categorical']
num_cols = column_groups['numeric']
calc_cols = column_groups['calc']

print(len(bin_cols), len(cat_cols), len(num_cols), len(calc_cols))

17 14 26 20


## 3. Evaluation setup
5-fold stratified cross-validation on the training part only; the test part stays untouched until notebook 04.
Main metric: normalised Gini (= 2·AUC − 1, the competition metric); also reported: ROC-AUC and PR-AUC
(average precision), which is more sensitive to the rare positive class.

In [4]:
# 3. Stratified 5-fold cross-validation and the metrics
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'roc_auc': 'roc_auc', 'pr_auc': 'average_precision'}

def evaluate(model, X, y, name):
    """Cross-validate a model and return Gini, ROC-AUC and PR-AUC (mean and std over folds)."""
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    gini = 2 * scores['test_roc_auc'] - 1
    return {
        'model': name,
        'gini_mean': gini.mean(),
        'gini_std': gini.std(),
        'roc_auc_mean': scores['test_roc_auc'].mean(),
        'pr_auc_mean': scores['test_pr_auc'].mean(),
        'pr_auc_std': scores['test_pr_auc'].std(),
        'fit_time_s': scores['fit_time'].mean(),
    }

## 4. Dummy baseline
The reference every model has to beat: it predicts the overall claim rate (3.6%) for every driver.

In [5]:
# 4. Dummy baseline: constant prediction
results = []
results.append(evaluate(DummyClassifier(strategy='prior'), X_train, y_train,
                        'Dummy (constant claim rate)'))
print(pd.DataFrame(results).round(4).to_string())

                         model  gini_mean  gini_std  roc_auc_mean  pr_auc_mean  pr_auc_std  fit_time_s
0  Dummy (constant claim rate)        0.0       0.0           0.5       0.0364         0.0      0.8728


**Observations:**
- The dummy model has Gini 0 (no ranking at all) and a PR-AUC equal to the claim rate (0.036).
  These are the lower bounds for every model in notebook 03.

**Next steps (to do):** preprocessing pipelines from the EDA log — for logistic regression: median imputation
with missing indicator, one-hot encoding with rare categories grouped, standardization, `ps_ind_14` dropped;
for tree models: NaN kept, no encoding needed.